# ETL Pipeline — Análisis de Desastres Naturales en México (2020–2025)

**Pregunta analítica:** ¿Qué cambios ocurrieron en el perfil de los desastres naturales en México después de la pandemia y qué estados experimentaron el mayor incremento en afectaciones?

**Tabla de hechos:** `fact_eventos_desastres`

**Fuente:** `https://www.datos.gob.mx/dataset/gestion_riesgos`

---
### Flujo del ETL
1. **Configuración** — credenciales y conexión a Aurora
2. **Catálogo y helpers** — normalización de estados y limpieza de datos
3. **Extract** — lectura desde tablas staging
4. **Transform** — limpieza y construcción del modelo dimensional
5. **Load** — carga de dimensiones y tabla de hechos 
6. **Validate** — validaciones post-carga
7. **Ejecutar ETL** — llamada principal
8. **Pruebas ETL** — Verificar los datos


## 0. Parámetros del proyecto
Reemplaza los valores de `AURORA_HOST`, `AURORA_DB` y `AURORA_PASSWORD` con los datos de tu instancia.

In [1]:
SCHEMA = "riesgos_proyecto"

HOST = "aurora-mod4.cluster-cmnqz9gfw97z.us-east-1.rds.amazonaws.com"
DATABASE = "northwind"
USER = "postgres"
PASSWORD = "Metzmellali"
AURORA_PORT = 5432

CONNECTION_STRING = (
    f"postgresql+psycopg2://{USER}:{PASSWORD}@{HOST}:{AURORA_PORT}/{DATABASE}"
)

## 1. Configuración de conexión a Aurora


In [2]:
from sqlalchemy import create_engine

engine = create_engine(
    CONNECTION_STRING,
    pool_pre_ping=True
)

print("✓ Conexión exitosa")

✓ Conexión exitosa


## 2. Imports y configuración de logging

In [3]:
#Lee las tres tablas staging cargadas en Aurora PostgreSQL, transforma los datos
#al modelo dimensional estrella y carga las dimensiones y la tabla de hechos.

import pandas as pd
import numpy as np
import logging
import re

from sqlalchemy import (
    create_engine,
    text
)

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s"
)

logger = logging.getLogger(__name__)

## 3. Catálogo de normalización


### Estados

Diccionario que mapea las variantes de nombres de estados

In [4]:
# Mapea variantes encontradas en los datos fuente al nombre oficial.
NORM_ESTADOS: dict[str, str] = {
    "AGUASCALIENTES": "Aguascalientes",
    "BAJA CALIFORNIA": "Baja California",
    "BAJA CALIFORNIA SUR": "Baja California Sur",
    "CAMPECHE": "Campeche",
    "CHIAPAS": "Chiapas",
    "CHIHUAHUA": "Chihuahua",
    "CIUDAD DE MEXICO": "Ciudad de México",
    "CIUDAD DE MÉXICO": "Ciudad de México",
    "CDMX": "Ciudad de México",
    "DISTRITO FEDERAL": "Ciudad de México",
    "D.F.": "Ciudad de México",
    "COAHUILA": "Coahuila",
    "COAHUILA DE ZARAGOZA": "Coahuila",
    "COLIMA": "Colima",
    "DURANGO": "Durango",
    "GUANAJUATO": "Guanajuato",
    "GUERRERO": "Guerrero",
    "HIDALGO": "Hidalgo",
    "JALISCO": "Jalisco",
    "MEXICO": "Estado de México",
    "ESTADO DE MEXICO": "Estado de México",
    "ESTADO DE MÉXICO": "Estado de México",
    "EDO. DE MEX.": "Estado de México",
    "MICHOACAN": "Michoacán",
    "MICHOACÁN": "Michoacán",
    "MICHOACAN DE OCAMPO": "Michoacán",
    "MORELOS": "Morelos",
    "NAYARIT": "Nayarit",
    "NUEVO LEON": "Nuevo León",
    "NUEVO LEÓN": "Nuevo León",
    "OAXACA": "Oaxaca",
    "PUEBLA": "Puebla",
    "QUERETARO": "Querétaro",
    "QUERÉTARO": "Querétaro",
    "QUINTANA ROO": "Quintana Roo",
    "SAN LUIS POTOSI": "San Luis Potosí",
    "SAN LUIS POTOSÍ": "San Luis Potosí",
    "SINALOA": "Sinaloa",
    "SONORA": "Sonora",
    "TABASCO": "Tabasco",
    "TAMAULIPAS": "Tamaulipas",
    "TLAXCALA": "Tlaxcala",
    "VERACRUZ": "Veracruz",
    "VERACRUZ DE IGNACIO DE LA LLAVE": "Veracruz",
    "YUCATAN": "Yucatán",
    "YUCATÁN": "Yucatán",
    "ZACATECAS": "Zacatecas",
}

# Estados válidos tras normalización (para filtrar registros inválidos)
ESTADOS_VALIDOS: set[str] = set(NORM_ESTADOS.values())

### Fenómenos

In [5]:
# FENOMENOS 
NORM_FENOMENOS = {

    "HIDROMETEOROLOGICO": "Hidrometeorológico",
    "HIDROMETEOROLÓGICO": "Hidrometeorológico",

    "GEOLOGICO": "Geológico",
    "GEOLÓGICO": "Geológico",

    "INCENDIO FORESTAL": "Incendio Forestal",

    "SIN DATO": None,
    "N/A": None,
    "": None
}

FENOMENOS_VALIDOS = {
    "Hidrometeorológico",
    "Geológico",
    "Incendio Forestal"
}

### Tipo de Evento

In [6]:
#TIPOS DE EVENTO 
NORM_TIPO_EVENTO = {
    "DESASTRE": "Desastre",
    "EMERGENCIA": "Emergencia"
}

TIPOS_EVENTO_VALIDOS = {
    "Desastre",
    "Emergencia"
}

## 4. Funciones de limpieza

Funciones para limpiar texto, número, moneda, estado, fenómeno y tipo de evento

In [7]:
def limpiar_texto(valor):
    if pd.isna(valor):
        return None
    return (
        str(valor)
        .strip()
        .upper()
    )


def limpiar_numero(valor):
    if pd.isna(valor):
        return np.nan
    valor = str(valor).replace(",", "")
    try:
        return float(valor)
    except:
        return np.nan


def limpiar_moneda(valor):
    if pd.isna(valor):
        return np.nan
    valor = str(valor)
    valor = valor.replace("$", "")
    valor = valor.replace(",", "")
    valor = valor.strip()
    if valor in ["", "-", "$-"]:
        return np.nan
    try:
        return float(valor)
    except:
        return np.nan


def normalizar_estado(valor):
    valor = limpiar_texto(valor)
    return NORM_ESTADOS.get(valor, valor)

def normalizar_fenomeno(valor):
    if pd.isna(valor):
        return None
    valor = str(valor).strip().upper()
    return NORM_FENOMENOS.get(valor, valor.title())

def normalizar_tipo_evento(valor):
    if pd.isna(valor):
        return None
    valor = str(valor).strip().upper()
    return NORM_TIPO_EVENTO.get(valor)

## 5. Extract — lectura desde Aurora

Lee las tres tablas staging desde Aurora. Todas las columnas vienen como TEXT — la conversión de tipos ocurre en Transform.

In [8]:
def extract(engine):

    logger.info("Leyendo tablas staging...")

    df_des = pd.read_sql("""
        SELECT *
        FROM riesgos_proyecto.stg_declaratorias_desastre
    """, engine)

    df_eme = pd.read_sql("""
        SELECT *
        FROM riesgos_proyecto.stg_declaratorias_emergencia
    """, engine)

    df_prev = pd.read_sql("""
        SELECT *
        FROM riesgos_proyecto.stg_proyectos_prevencion
    """, engine)

    logger.info(f"Desastres: {len(df_des):,}")
    logger.info(f"Emergencias: {len(df_eme):,}")
    logger.info(f"Prevención: {len(df_prev):,}")

    return df_des, df_eme, df_prev

## 6. Transform — limpieza y construcción del modelo

Limpia cada fuente por separado y luego las consolida en el grano de la fact: `(estado, año, tipo_fenomeno, tipo evento, municipios)`.

In [9]:
def transform_desastres(df_des):

    logger.info("Transformando declaratorias de desastre...")

    df = df_des.copy()

    # Estado
    df["estado"] = df["entidad_federativa"].apply(
        normalizar_estado
    )

    # Año
    df["anio"] = pd.to_numeric(
        df["anio"],
        errors="coerce"
    )

    # Fenómeno
    df["fenomeno"] = df["tipo_fenomeno"].apply(
        normalizar_fenomeno
    )

    # Tipo evento
    df["tipo_evento"] = "Desastre"

    # Municipios afectados
    df["municipios_afectados"] = pd.to_numeric(
        df["numero_municipios_corroborados"],
        errors="coerce"
    ).fillna(0)

    # Métricas inexistentes en esta tabla
    df["poblacion_afectada"] = 0
    df["poblacion_atendida"] = 0
    df["costo_total"] = 0

    # Cada fila representa un evento
    df["cantidad_eventos"] = 1

    df = df[
        df["estado"].isin(ESTADOS_VALIDOS)
    ]

    logger.info(
        f"Desastres transformados: {len(df):,}"
    )

    return df[[
        "estado",
        "anio",
        "fenomeno",
        "tipo_evento",
        "municipios_afectados",
        "poblacion_afectada",
        "poblacion_atendida",
        "costo_total",
        "cantidad_eventos"
    ]]



def transform_emergencias(df_eme):

    logger.info("Transformando declaratorias de emergencia...")

    df = df_eme.copy()

    # Estado
    df["estado"] = df["entidad_federativa"].apply(
        normalizar_estado
    )

    # Año
    df["anio"] = pd.to_numeric(
        df["anio_ocurrencia_evento"],
        errors="coerce"
    )

    # Fenómeno
    df["fenomeno"] = df["tipo_fenomeno"].apply(
        normalizar_fenomeno
    )

    # Tipo evento
    df["tipo_evento"] = "Emergencia"

    # Municipios afectados
    df["municipios_afectados"] = pd.to_numeric(
        df["numero_municipios_corroborados"],
        errors="coerce"
    ).fillna(0)

    # Población afectada
    df["poblacion_afectada"] = (
        df["poblacion_afectada"]
        .apply(limpiar_numero)
    )

    # Población atendida
    df["poblacion_atendida"] = (
        df["poblacion_atendida"]
        .apply(limpiar_numero)
    )

    # Costo total
    df["costo_total"] = (
        df["costo_total_declaratoria"]
        .apply(limpiar_moneda)
    )

    df["cantidad_eventos"] = 1

    df = df[
        df["estado"].isin(ESTADOS_VALIDOS)
    ]

    logger.info(
        f"Emergencias transformadas: {len(df):,}"
    )

    return df[[
        "estado",
        "anio",
        "fenomeno",
        "tipo_evento",
        "municipios_afectados",
        "poblacion_afectada",
        "poblacion_atendida",
        "costo_total",
        "cantidad_eventos"
    ]]


def transform_proyectos(df_prev):

    logger.info("Transformando proyectos de prevención...")

    df = df_prev.copy()

    df["estado"] = (
        df["entidad_federativa_solicitante"]
        .apply(normalizar_estado)
    )

    df["anio_autorizacion"] = pd.to_numeric(
        df["anio_autorizacion"],
        errors="coerce"
    )

    df["personas_beneficiadas"] = (
        df["personas_beneficiadas"]
        .apply(limpiar_numero)
    )

    df["costo_total"] = (
        df["costo_total_mxn_peso"]
        .apply(limpiar_moneda)
    )

    logger.info(
        f"Proyectos transformados: {len(df):,}"
    )

    return df

In [10]:
def build_fact(
    df_des,
    df_eme,
    df_prev
):

    logger.info(
        "Construyendo tabla de hechos..."
    )

    fact = pd.concat(
        [df_des, df_eme],
        ignore_index=True
    )

    fact = fact[
        fact["anio"].between(
            2018,
            2025
        )
    ]

    logger.info(
        f"Fact construida: {len(fact):,} registros"
    )

    return fact

In [11]:
def build_dim_estado(fact):

    return pd.DataFrame({
        "estado":
        sorted(
            fact["estado"]
            .dropna()
            .unique()
        )
    })

def build_dim_tiempo(fact):

    return pd.DataFrame({
        "anio":
        sorted(
            fact["anio"]
            .dropna()
            .unique()
        )
    })

def build_dim_fenomeno(fact):

    return pd.DataFrame({
        "fenomeno":
        sorted(
            fact["fenomeno"]
            .dropna()
            .unique()
        )
    })


def build_dim_tipo_evento():

    return pd.DataFrame({
        "tipo_evento": [
            "Desastre",
            "Emergencia"
        ]
    })

## 7. Load — carga de dimensiones y fact

In [37]:
def load_dim_estado(fact, engine):

    logger.info("Cargando dim_estado...")

    dim_estado = (
        pd.DataFrame({
            "estado": sorted(
                fact["estado"]
                .dropna()
                .unique()
            )
        })
        .reset_index(drop=True)
    )

    dim_estado["id_estado"] = (
        dim_estado.index + 1
    )

    dim_estado = dim_estado[
        ["id_estado", "estado"]
    ]

    # Limpia la dimensión para recarga
    with engine.begin() as conn:
        conn.execute(
            text("""
            TRUNCATE TABLE riesgos_proyecto.dim_estado
            RESTART IDENTITY CASCADE
            """)
    )

    dim_estado.to_sql(
        "dim_estado",
        engine,
        schema="riesgos_proyecto",
        if_exists="append",
        index=False,
        method="multi"
    )

    logger.info(
        f"✓ {len(dim_estado)} estados cargados"
    )

    return dim_estado

def load_dim_tiempo(fact, engine):

    logger.info("Cargando dim_tiempo...")

    dim_tiempo = (
        pd.DataFrame({
            "anio": sorted(
                fact["anio"]
                .dropna()
                .unique()
            )
        })
        .reset_index(drop=True)
    )

    dim_tiempo["id_tiempo"] = (
        dim_tiempo.index + 1
    )

    dim_tiempo = dim_tiempo[
        ["id_tiempo", "anio"]
    ]

    with engine.begin() as conn:
        conn.execute(
            text("""
            TRUNCATE TABLE riesgos_proyecto.dim_tiempo
            RESTART IDENTITY CASCADE
            """)
    )

    dim_tiempo.to_sql(
        "dim_tiempo",
        engine,
        schema="riesgos_proyecto",
        if_exists="append",
        index=False,
        method="multi"
    )

    logger.info(
        f"✓ {len(dim_tiempo)} años cargados"
    )

    return dim_tiempo


def load_dim_fenomeno(fact, engine):

    logger.info("Cargando dim_fenomeno...")

    dim_fenomeno = (
        pd.DataFrame({
            "tipo_fenomeno": sorted(
                fact["fenomeno"]
                .dropna()
                .unique()
            )
        })
        .reset_index(drop=True)
    )

    dim_fenomeno["id_fenomeno"] = (
        dim_fenomeno.index + 1
    )

    dim_fenomeno = dim_fenomeno[
        ["id_fenomeno", "tipo_fenomeno"]
    ]

    with engine.begin() as conn:
        conn.execute(
            text("""
            TRUNCATE TABLE riesgos_proyecto.dim_fenomeno
            RESTART IDENTITY CASCADE
            """)
    )

    dim_fenomeno.to_sql(
        "dim_fenomeno",
        engine,
        schema="riesgos_proyecto",
        if_exists="append",
        index=False,
        method="multi"
    )

    logger.info(
        f"✓ {len(dim_fenomeno)} fenómenos cargados"
    )

    return dim_fenomeno

def load_dim_tipo_evento(engine):

    logger.info("Cargando dim_tipo_evento...")

    dim_tipo_evento = pd.DataFrame({

        "id_tipo_evento": [1, 2],

        "tipo_evento": [
            "Desastre",
            "Emergencia"
        ]
    })

    with engine.begin() as conn:
        conn.execute(
            text("""
            TRUNCATE TABLE riesgos_proyecto.dim_tipo_evento
            RESTART IDENTITY CASCADE
            """)
    )

    dim_tipo_evento.to_sql(
        "dim_tipo_evento",
        engine,
        schema="riesgos_proyecto",
        if_exists="append",
        index=False,
        method="multi"
    )

    logger.info(
        "✓ 2 tipos de evento cargados"
    )

    return dim_tipo_evento


In [41]:
def load_fact(
    fact,
    dim_estado,
    dim_tiempo,
    dim_fenomeno,
    dim_tipo_evento,
    engine
):

    logger.info("Cargando fact_eventos_desastres...")

    # Estado
    fact = fact.merge(
        dim_estado,
        on="estado",
        how="left"
    )

    # Tiempo
    fact = fact.merge(
        dim_tiempo,
        on="anio",
        how="left"
    )

    # Fenómeno
    fact = fact.merge(
        dim_fenomeno,
        left_on="fenomeno",
        right_on="tipo_fenomeno",
        how="left"
    )

    # Tipo Evento
    fact = fact.merge(
        dim_tipo_evento,
        on="tipo_evento",
        how="left"
    )

    fact_final = fact[[
        "id_estado",
        "id_tiempo",
        "id_fenomeno",
        "id_tipo_evento",
        "municipios_afectados",
        "poblacion_afectada",
        "poblacion_atendida",
        "costo_total",
        "cantidad_eventos"
    ]]

    with engine.begin() as conn:
        conn.execute(
            text("""
            TRUNCATE TABLE riesgos_proyecto.fact_eventos_desastres
            RESTART IDENTITY
            """)
        )

    fact_final.to_sql(
        "fact_eventos_desastres",
        engine,
        schema="riesgos_proyecto",
        if_exists="append",
        index=False,
        method="multi",
        chunksize=1000
    )

    logger.info(
        f"✓ {len(fact_final):,} registros cargados en FACT"
    )

## 8. Validaciones post-carga

Verifica integridad referencial y ausencia de valores negativos en las métricas.

In [42]:
from sqlalchemy import text
"""
    Validaciones post-carga:
    1. Conteos por tabla.
    2. Integridad referencial (FKs no nulas).
    3. Ausencia de valores negativos en métricas.
    """
def validate(engine):
    """
    Validaciones post-carga:

    1. Conteos por tabla.
    2. Integridad referencial.
    3. Ausencia de valores negativos.
    4. Resumen de métricas para análisis.
    """

    logger.info("Validaciones post-carga...")

    # =====================================================
    # TOP 10 ESTADO-AÑO CON MÁS EVENTOS
    # =====================================================

    resumen = pd.read_sql(text("""
        SELECT
            de.estado,
            dt.anio,
            COUNT(*) AS total_eventos,
            SUM(COALESCE(fe.municipios_afectados,0)) AS municipios_afectados,
            SUM(COALESCE(fe.poblacion_afectada,0)) AS poblacion_afectada,
            SUM(COALESCE(fe.costo_total,0)) AS costo_total
        FROM riesgos_proyecto.fact_eventos_desastres fe
        JOIN riesgos_proyecto.dim_estado de
            ON fe.id_estado = de.id_estado
        JOIN riesgos_proyecto.dim_tiempo dt
            ON fe.id_tiempo = dt.id_tiempo
        GROUP BY de.estado, dt.anio
        ORDER BY total_eventos DESC
        LIMIT 10
    """), engine)

    logger.info(
        "Top 10 Estado-Año por número de eventos:\n%s",
        resumen.to_string(index=False)
    )

    # =====================================================
    # FK NULAS
    # =====================================================

    nulls = pd.read_sql(text("""
        SELECT COUNT(*) AS n
        FROM riesgos_proyecto.fact_eventos_desastres
        WHERE id_estado IS NULL
           OR id_tiempo IS NULL
           OR id_fenomeno IS NULL
           OR id_tipo_evento IS NULL
    """), engine).iloc[0,0]

    assert nulls == 0, (
        f"Hay {nulls} registros con FK nulas"
    )

    logger.info("✓ Sin FK nulas")

    # =====================================================
    # VALORES NEGATIVOS
    # =====================================================

    negatives = pd.read_sql(text("""
        SELECT COUNT(*) AS n
        FROM riesgos_proyecto.fact_eventos_desastres
        WHERE municipios_afectados < 0
           OR poblacion_afectada < 0
           OR poblacion_atendida < 0
           OR costo_total < 0
           OR cantidad_eventos < 0
    """), engine).iloc[0,0]

    assert negatives == 0, (
        f"Hay {negatives} registros con valores negativos"
    )

    logger.info("✓ Sin valores negativos")

    # =====================================================
    # RESUMEN FINAL
    # =====================================================

    totales = pd.read_sql(text("""
        SELECT
            COUNT(*) AS registros_fact,
            SUM(cantidad_eventos) AS total_eventos,
            SUM(COALESCE(municipios_afectados,0))
                AS municipios_afectados,
            ROUND(
                SUM(COALESCE(poblacion_afectada,0)),
                0
            ) AS poblacion_afectada,
            ROUND(
                SUM(COALESCE(costo_total,0)),
                2
            ) AS costo_total
        FROM riesgos_proyecto.fact_eventos_desastres
    """), engine)

    logger.info(
        "Resumen final fact_eventos_desastres:\n%s",
        totales.to_string(index=False)
    )

    logger.info("✓ Validaciones completadas")

## 9. Ejecutar el ETL completo

Orquesta el flujo completo: Extract → Transform → Load → Validate. 

In [43]:
from sqlalchemy import create_engine

engine = create_engine(
    CONNECTION_STRING,
    pool_pre_ping=True
)

try:

    # =====================================================
    # EXTRACT
    # =====================================================

    logger.info("EXTRACT")

    df_des_raw, df_eme_raw, df_prev_raw = extract(engine)

    # =====================================================
    # TRANSFORM
    # =====================================================

    logger.info("TRANSFORM")

    df_des = transform_desastres(df_des_raw)

    df_eme = transform_emergencias(df_eme_raw)

    df_prev = transform_proyectos(df_prev_raw)

    fact = build_fact(
        df_des,
        df_eme,
        df_prev
    )

    # =====================================================
    # LOAD DIMENSIONES
    # =====================================================

    logger.info("LOAD DIMENSIONES")

    dim_estado = load_dim_estado(
        fact,
        engine
    )

    dim_tiempo = load_dim_tiempo(
        fact,
        engine
    )

    dim_fenomeno = load_dim_fenomeno(
        fact,
        engine
    )

    dim_tipo_evento = load_dim_tipo_evento(
        engine
    )

    # =====================================================
    # LOAD FACT
    # =====================================================

    logger.info("LOAD FACT")

    load_fact(
        fact,
        dim_estado,
        dim_tiempo,
        dim_fenomeno,
        dim_tipo_evento,
        engine
    )

    # =====================================================
    # VALIDATE
    # =====================================================

    validate(engine)

    print("\n✅ ETL COMPLETADO EXITOSAMENTE")

except AssertionError as ae:

    print(
        f"\n❌ Validación fallida: {ae}"
    )

except Exception as exc:

    import traceback

    print(
        f"\n❌ ETL falló: {exc}"
    )

    traceback.print_exc()

finally:

    engine.dispose()

    print("\nConexión cerrada")

2026-06-10 20:56:39,109 - INFO - EXTRACT
2026-06-10 20:56:39,110 - INFO - Leyendo tablas staging...
2026-06-10 20:56:43,462 - INFO - Desastres: 122
2026-06-10 20:56:43,463 - INFO - Emergencias: 194
2026-06-10 20:56:43,464 - INFO - Prevención: 34
2026-06-10 20:56:43,465 - INFO - TRANSFORM
2026-06-10 20:56:43,465 - INFO - Transformando declaratorias de desastre...
2026-06-10 20:56:43,469 - INFO - Desastres transformados: 122
2026-06-10 20:56:43,471 - INFO - Transformando declaratorias de emergencia...
2026-06-10 20:56:43,479 - INFO - Emergencias transformadas: 192
2026-06-10 20:56:43,480 - INFO - Transformando proyectos de prevención...
2026-06-10 20:56:43,481 - INFO - Proyectos transformados: 34
2026-06-10 20:56:43,482 - INFO - Construyendo tabla de hechos...
2026-06-10 20:56:43,484 - INFO - Fact construida: 312 registros
2026-06-10 20:56:43,484 - INFO - LOAD DIMENSIONES
2026-06-10 20:56:43,485 - INFO - Cargando dim_estado...
2026-06-10 20:56:44,487 - INFO - ✓ 27 estados cargados
2026-0


✅ ETL COMPLETADO EXITOSAMENTE

Conexión cerrada


In [44]:
logger.info(
    f"""
    ========= MÉTRICAS DE CALIDAD =========

    Desastres leídos: {len(df_des_raw):,}
    Emergencias leídas: {len(df_eme_raw):,}

    Registros fact: {len(fact):,}

    Estados únicos: {fact['estado'].nunique()}
    Fenómenos únicos: {fact['fenomeno'].nunique()}

    =======================================
    """
)

2026-06-10 20:57:57,351 - INFO - 
    ========= MÉTRICAS DE CALIDAD =========

    Desastres leídos: 122
    Emergencias leídas: 194

    Registros fact: 312

    Estados únicos: 27
    Fenómenos únicos: 3

    


In [45]:
logger.info(
    f"""
    ========= MÉTRICAS DE CALIDAD =========

    Desastres leídos: {len(df_des_raw):,}
    Emergencias leídas: {len(df_eme_raw):,}
    Proyectos prevención leídos: {len(df_prev_raw):,}

    Registros fact: {len(fact):,}

    Estados únicos: {fact['estado'].nunique()}
    Fenómenos únicos: {fact['fenomeno'].nunique()}
    Tipos evento únicos: {fact['tipo_evento'].nunique()}

    Municipios afectados total: {fact['municipios_afectados'].sum():,.0f}

    Población afectada total: {fact['poblacion_afectada'].sum():,.0f}

    Población atendida total: {fact['poblacion_atendida'].sum():,.0f}

    Costo total registrado: ${fact['costo_total'].sum():,.2f}

    =======================================
    """
)

2026-06-10 20:59:17,842 - INFO - 
    ========= MÉTRICAS DE CALIDAD =========

    Desastres leídos: 122
    Emergencias leídas: 194
    Proyectos prevención leídos: 34

    Registros fact: 312

    Estados únicos: 27
    Fenómenos únicos: 3
    Tipos evento únicos: 2

    Municipios afectados total: 2,976

    Población afectada total: 3,190,614

    Población atendida total: 2,777,905

    Costo total registrado: $3,977,964,148.50

    


In [46]:
logger.info(
    f"""
    ========= INTEGRIDAD =========

    Estados nulos: {fact['estado'].isna().sum()}
    Años nulos: {fact['anio'].isna().sum()}
    Fenómenos nulos: {fact['fenomeno'].isna().sum()}
    Tipos evento nulos: {fact['tipo_evento'].isna().sum()}

    ==============================
    """
)

2026-06-10 20:59:39,734 - INFO - 
    ========= INTEGRIDAD =========

    Estados nulos: 0
    Años nulos: 0
    Fenómenos nulos: 0
    Tipos evento nulos: 0

    


## Prueba del Modelo Dimensional

In [47]:
query_validacion = """
SELECT
    de.estado,
    dt.anio,
    df.tipo_fenomeno,
    dte.tipo_evento,
    SUM(f.cantidad_eventos) AS total_eventos
FROM riesgos_proyecto.fact_eventos_desastres f
JOIN riesgos_proyecto.dim_estado de
    ON f.id_estado = de.id_estado
JOIN riesgos_proyecto.dim_tiempo dt
    ON f.id_tiempo = dt.id_tiempo
JOIN riesgos_proyecto.dim_fenomeno df
    ON f.id_fenomeno = df.id_fenomeno
JOIN riesgos_proyecto.dim_tipo_evento dte
    ON f.id_tipo_evento = dte.id_tipo_evento
GROUP BY
    de.estado,
    dt.anio,
    df.tipo_fenomeno,
    dte.tipo_evento
ORDER BY total_eventos DESC
LIMIT 20
"""

df_prueba = pd.read_sql(
    query_validacion,
    engine
)

df_prueba

,estado,anio,tipo_fenomeno,tipo_evento,total_eventos
0,Chiapas,2020,Hidrometeorológico,Emergencia,11
1,Veracruz,2020,Hidrometeorológico,Emergencia,10
2,Oaxaca,2020,Hidrometeorológico,Emergencia,10
3,Oaxaca,2020,Geológico,Emergencia,9
4,Chiapas,2019,Hidrometeorológico,Emergencia,8
5,Tabasco,2020,Hidrometeorológico,Emergencia,7
6,Durango,2020,Hidrometeorológico,Emergencia,7
7,Sonora,2019,Hidrometeorológico,Emergencia,6
8,Chiapas,2020,Hidrometeorológico,Desastre,6
9,Oaxaca,2021,Hidrometeorológico,Emergencia,6
